In [ ]:
!pip install -U langchain-community pypdf


In [29]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    file_path = "./documents/Prof_C.V.Jawahars_Notes.pdf",
    extract_images = True
)

In [31]:
!pip install pypdf rapidocr-onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 77.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 66.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [rapidocr-onnxruntime]ncv-python]

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [32]:
from langchain_community.document_loaders import PyPDFLoader

# 1. Initialize the loader with image extraction enabled
loader = PyPDFLoader(
    file_path="./documents/Prof_C.V.Jawahars_Notes.pdf",
    extract_images=True
)

# 2. Load and parse the PDF pages
print("Loading PDF and extracting text from images (this might take a moment)...")
pages = loader.load()

# 3. Verify the output
print(f"Successfully loaded {len(pages)} pages.")

# Print a snippet of the first page to check the content
if pages:
    print("\n--- Sample Content from Page 1 ---")
    print(pages[0].page_content[:500])  # Prints the first 500 characters

Loading PDF and extracting text from images (this might take a moment)...
Successfully loaded 10 pages.

--- Sample Content from Page 1 ---
Overview of Moderm AI and ML Chapter 1
Introduction
C.V Jawahar
1.1 Introduction
Recent years had seen the success of AI in various walks of our life. It looks like it is going to aﬀect in much
deeper manner in the coming years. What is really happening? In the simplest form, we have found ways
to come up with acceptable solutions for a large number of problems that could impact our day to day life.
Why is it happening? There are many answers to this. In short, our algorithmic understanding on h


In [33]:
embedding_model = "embeddinggemma:300m-qat-q4_0"
llm_model = "mistral:latest"

In [34]:
!pip install -U langchain-text-splitters


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [35]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# (Assuming these variables are defined elsewhere in your code)


# 1. Initialize Local Ollama Embeddings and LLM
print("Initializing Ollama models...")
embeddings = OllamaEmbeddings(model=embedding_model)
llm = ChatOllama(model=llm_model, temperature=0)

# 2. Load and Split Documents
print("Loading and splitting documents...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = []
for doc in pages:
    chunks = text_splitter.split_text(doc.page_content)
    for i, chunk in enumerate(chunks):
        documents.append(Document(
            page_content=chunk,
            metadata={"source": f"{doc.metadata.get('source', 'unknown')}_chunk_{i}"}
        ))
print(f"Total document chunks created: {len(documents)}")

# 3. Create Chroma Vector Store and Add Documents
print("Creating Chroma vector store and adding documents...")
# Corrected initialization: Pass documents directly to the classmethod, or use from_documents
vector_store = Chroma.from_documents(
    documents=documents, 
    embedding=embeddings, 
    collection_name="rag_collection"
)
print("Documents added to vector store.")

# 4. Define RetrievalQA Chain (Using Modern LCEL)
print("Setting up RAG chain...")
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Define a standard RAG prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

# Build the LCEL Chain
# We use a RunnablePassthrough.assign to pass source docs through to the final output
rag_chain = (
    RunnablePassthrough.assign(context=lambda x: retriever.invoke(x["question"]))
    | RunnablePassthrough.assign(answer=prompt | llm | StrOutputParser())
)
print("RAG chain is ready.")



Initializing Ollama models...
Loading and splitting documents...
Total document chunks created: 35
Creating Chroma vector store and adding documents...
Documents added to vector store.
Setting up RAG chain...
RAG chain is ready.


In [36]:
# 5. Test the RAG System with a Sample Query
query = "What is a regression problem ?"
print(f"Query: {query}\n")

# Invoke the chain
result = rag_chain.invoke({"question": query})

print("Answer:")
print(result['answer'])

print("\nSource Documents:")
for doc in result['context']:
    print(f"- {doc.metadata['source']}")

Query: What is a regression problem ?

Answer:
 A regression problem, as described in the provided context, is a type of statistical modeling where the dependent variable (yi) is a real quantity. It could be a real vector or a simple real number. The goal of regression is to establish relationships between one or more independent variables and the dependent variable, with the aim of predicting the value of the dependent variable based on the values of the independent variables. In this context, it's mentioned that discussions about classification (which involves categorical data) can be adapted for regression problems with some minor changes.

Source Documents:
- ./documents/Prof_C.V.Jawahars_Notes.pdf_chunk_3
- ./documents/Prof_C.V.Jawahars_Notes.pdf_chunk_3
- ./documents/Prof_C.V.Jawahars_Notes.pdf_chunk_3
